In [1]:
!pip install --quiet --upgrade "google-cloud-aiplatform[agent_engines,adk]"
!pip install googlemaps --q

print("Installation complete.")

Installation complete.


In [2]:
import os
import requests
from typing import Dict, Optional
from google.adk.agents import Agent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
import warnings
import logging
import json
import vertexai

from google.genai import types
from google.adk.tools import AgentTool
from google.adk.tools import google_search
from google.adk.tools import FunctionTool

In [3]:
AGENT_MODEL = "gemini-2.5-flash"
API_KEY = ""
STAGING_BUCKET="gs://adk-capstone"

vertexai.init(
    project="qwiklabs-gcp-00-819951dd6b01",
    location="us-central1",
    staging_bucket="gs://adk-capstone"
)


**TOOLS**

In [4]:
import googlemaps
from typing import Optional, Tuple

def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """Converts a physical address to latitude and longitude using Google's Geocoding API.

    Args:
        api_key: Your Google Maps API key with the Geocoding API enabled.
        address: The human-readable address or place name to geocode.

    Returns:
        A tuple containing the latitude and longitude, or None if an error occurs.
    """
    # Initialize the client with your API key
    gmaps = googlemaps.Client(key=API_KEY)

    try:
        # Make the API call to geocode the address
        geocode_result = gmaps.geocode(location)

        if not geocode_result:
            print(f"Warning: No results found for address: '{location}'")
            return None

        location = geocode_result[0]['geometry']['location']
        latitude = location['lat']
        longitude = location['lng']

        return (latitude, longitude)

    except googlemaps.exceptions.ApiError as e:
        print(f"An API error occurred: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

print(get_lat_lon("New York, NY"))

(40.7127753, -74.0059728)


In [5]:
import googlemaps
from typing import Optional, Dict

def get_directions(origin: str, destination: str) -> Optional[Dict]:
    """
    Gets directions between two locations using the Google Maps Directions API.

    Args:
        origin: The starting point (address or place name).
        destination: The ending point (address or place name).

    Returns:
        A dictionary containing the directions, or None if an error occurs.
    """
    gmaps = googlemaps.Client(key=API_KEY)

    try:
        directions_result = gmaps.directions(origin, destination)

        if not directions_result:
            print(f"Warning: No directions found between '{origin}' and '{destination}'")
            return None

        return directions_result

    except googlemaps.exceptions.ApiError as e:
        print(f"An API error occurred: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

print(get_directions("New York, NY", "Los Angeles, CA"))

[{'bounds': {'northeast': {'lat': 41.7568586, 'lng': -74.0062188}, 'southwest': {'lat': 34.0549066, 'lng': -118.2426603}}, 'copyrights': 'Powered by Google, ©2026 Google', 'legs': [{'distance': {'text': '2,789 mi', 'value': 4489190}, 'duration': {'text': '1 day 17 hours', 'value': 147322}, 'end_address': 'Los Angeles, CA, USA', 'end_location': {'lat': 34.0549066, 'lng': -118.2426603}, 'start_address': 'New York, NY, USA', 'start_location': {'lat': 40.7124787, 'lng': -74.0062188}, 'steps': [{'distance': {'text': '200 ft', 'value': 61}, 'duration': {'text': '1 min', 'value': 21}, 'end_location': {'lat': 40.71213280000001, 'lng': -74.0056993}, 'html_instructions': 'Head <b>southeast</b> toward <b>Park Row</b><div style="font-size:0.9em">Partial restricted usage road</div>', 'polyline': {'points': '_tnwFziubM^cAFS@A@CDCRE'}, 'start_location': {'lat': 40.7124787, 'lng': -74.0062188}, 'travel_mode': 'DRIVING'}, {'distance': {'text': '0.1 mi', 'value': 226}, 'duration': {'text': '1 min', 'val

In [9]:
def get_nws_weather(lat: float, lng: float) -> str:
    """
    Retrieves the weather forecast from the National Weather Service.
    Args:
        lat (float): Latitude
        lng (float): Longitude
    Returns:
        str: The latest weather forecast.
    """
    import requests

    # NWS requires a two-step process: /points to get the grid, then /forecast
    points_url = f"https://api.weather.gov/points/{lat},{lng}"
    headers = {'User-Agent': '(myweatheragent.com, tim.r.spicer@gmail.com)'}

    res = requests.get(points_url, headers=headers).json()
    #print(res)
    forecast_url = res['properties']['forecast']

    forecast_res = requests.get(forecast_url, headers=headers).json()
    periods = forecast_res['properties']['periods']
    return periods[0]['detailedForecast']

**AGENTS**

In [30]:
weather_agent = Agent(
    name="weather_agent_v1",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Provides weather information for specific cities.",
    instruction="""You are a helpful weather assistant. When the user asks for the weather in a city,
     use the (lat, lon) pair and the get_nws_weather tool with those cooridnates.""",
    tools=[get_nws_weather], # Pass the function directly
)

In [31]:
google_search_agent = Agent(
    name="search_agent_v1",
    model=AGENT_MODEL,
    description="Searches the web for information about a specific topic.",
    instruction="""You are a helpful research assistant. Perform a google search to gather the most relevant information for the user.""",
    tools=[google_search],
)

news_search_agent = Agent(
    name="news_search_agent",
    model=AGENT_MODEL,
    output_key="recent_news",
    description="Useful for finding current news on disasters",
    instruction="""
    You are a research specialist for weather and disaster news.
    Use Google Search to find accurate information based on the user's location.
    """,
    tools=[google_search],
)

In [32]:
maps_agent = Agent(
    name="maps_agent",
    model=AGENT_MODEL,
    output_key="lat_long",
    description="Provides location in latitude and longitude",
    instruction="""
    You receive a city or zip code location and convert it into latitude and longitude
    using the 'get_coordinates' tool. Your response should only include the latitude and longitude
    in the format (latitude, longitude).
    """,
    tools=[get_lat_lon],
)

routing_agent = Agent(
    name="routing_agent",
    model=AGENT_MODEL,
    output_key="routing directions",
    description="Provides direction from user's latitude and longitude",
    instruction="""
    You will receive a location and provide major evacuation routes.
    Use tool 'get_directions' to get a detailed procedure for evacuation.
    """,
    tools=[get_directions]
)

In [33]:
realtime_agent = SequentialAgent(
    name="realtime_agent",
    sub_agents=[maps_agent, weather_agent, news_search_agent, routing_agent],
    description="Emergency response coordinator for FEMA disaster assistance.",
)

In [34]:
critique_agent = Agent(
    name="critique_agent",
    model=AGENT_MODEL,
    description="Evaluates search results and provides improvement suggestions.",
    instruction="""
    You are a critique agent. Your role is to evaluate the quality and relevance of the output from the 'Search' agent.
    Identify any missing information or areas for improvement in the search results or the generated response.
    Provide constructive suggestions for refining the search results or the generated response.
    """,
)

refine_agent = Agent(
    name="refine_agent",
    model=AGENT_MODEL,
    description="Rewrites responses based on search results and critique suggestions.",
    instruction="""
    You are a refine agent. Your role is to rewrite and improve the response.
    Carefully consider the original search results and the suggestions provided by the 'Critique' agent.
    Integrate the feedback to produce a more accurate, comprehensive, and well-structured final response.
    """,
)

In [35]:
refinement_agent = SequentialAgent(
    name="refinement_agent_v1",
    description="Validates and refines responses from the agents to the user, and returns that repsonse to the main agent.",
    sub_agents=[
        critique_agent,
        refine_agent,
    ]
)

In [36]:
main_agent = Agent(
    name="main_agent",
    model=AGENT_MODEL,
    description="Emergency response coordinator for FEMA disaster assistance.",
    instruction="""
    You are a FEMA emergency assistant. Your goal is to provide local disaster-related info for civilians.

    1. Greeting: If the user says a simple greeting, respond politely, state your purpose, and ask for their current city or zip code.
    2. Location Acquisition: If the location is missing, you MUST ask the user for it before proceeding.
    3. Sequence Execution: Once a location is provided, execute the 'realtime_agent'.
    4. The final response to the user must be fed through the 'refinement_agent' before being returned; this agent is used only as a processor, not for direct
    conversation.

    ### Response Guidelines:
    - Summarize the weather, news, and routes with concise bullet points.
    - Scope: Use 'google_search_agent' only for specific questions that fall outside the scope of weather, news, or maps.

    ### Constraints:
    - Do not provide weather or disaster info for locations outside the United States.
    - If any sub-agent fails, inform the user but continue with the remaining steps.
    """,
    sub_agents=[realtime_agent,google_search_agent, refinement_agent]
)

In [37]:
session_service = InMemorySessionService()

# Define constants for identifying the interaction context
APP_NAME = "FEMA_helper"
USER_ID = "user_1"
SESSION_ID = "session_001" # Using a fixed ID for simplicity

await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(
    agent=main_agent, # The agent we want to run
    app_name=APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)

Session created: App='FEMA_helper', User='user_1', Session='session_001'


In [38]:
import vertexai

vertexai.init(
    project="qwiklabs-gcp-00-819951dd6b01",
    location="us-central1",
    staging_bucket="gs://adk-capstone"
)

from vertexai.preview.reasoning_engines import AdkApp
app = AdkApp(agent=main_agent)
for event in app.stream_query(
user_id="USER_ID",
message="I'm a resident of MD and heard about an impending blizzard. Is this true?",
):
  print(event)

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:844: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-689e58f1-dec0-450c-93b6-42fb80bf71b4', 'args': {'agent_name': 'realtime_agent'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'Co4DAY89a1-7mRwKWejdrgOdHwrl4y_zaR3kM-nvnjm9wyWmLM5rIPiQg7YCLdELymmfwXcSlZuxakKbdr4bCM1GuNE6cSV5Pf8tevH2X3ssY0gRPmFLT-GXS6Qn37qpEwmKCTpWsr82nFgolk8B2_3m9s5XbN0FuBg4LGf8xnBosf94natgwwlPAsy6Cl4X9z1gmh9iA_JbCkwCfgpK1bw8hj_5sikmWrGFVyukVlcYOY8pyO9tjbPM7_ylty4TC12oOtSdE5uhpKI8DLHplufk42JSqSkxtV8LN2fhgCWlJpkFQZoAsVOPxfpXmrH3Anx92Zq-SGMqZV3IdLV6VKReWsobSffgaijMb34ntz71Q_n33tYwMCzGgpu74PixxfOHfrB4ouvQnyAb-dDHhop450tJeSXbACP5QPrMyECKV7B_EfxSXR55s8Rmxa6pVXrWWR--dDj61Hei2kswz-9tQ58NDi_dUmi9y1dMeXm8gns3xHxtD6Ph5m3_ZDsowCNRzDs8mCAv7y9p1OfkokU='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 12, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 12}], 'prompt_token_count': 625, 'prompt_tokens_details': [{'modality': 'TEXT'

In [185]:
!adk --version

adk, version 1.23.0


In [189]:
!pip install googlemaps

In [39]:
from vertexai import agent_engines
remote_agent = agent_engines.create(
    app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]", "litellm", "google-adk[extensions]", "googlemaps"]
)

INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.139.0', 'pydantic': '2.12.3'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'litellm', 'google-adk[extensions]', 'googlemaps', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket adk-capstone
INFO:vertexai.agent_engines:Wrote to gs://adk-capstone/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://adk-capstone/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://adk-capstone/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/579193723182/locations/us-cen